In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import psycopg2
from sqlalchemy import create_engine
from sqlalchemy import inspect
import re
import telebot
TOKEN = "8741114144:AAGUGuqLnmWYPzZ1Z3Ol997ogsg7J-CNUew"
bot = telebot.TeleBot(TOKEN)

# ---------- Данные бота ----------
courses = ["Python", "SQL", "Анализ данных"]
CONTACTS = ("Учебный центр", "+7 700 000 00 00")
users = set()
HOURS = {"Python": 30, "SQL": 24, "Анализ данных": 36}
SCHEDULE = {
    "Понедельник": ["Python", "SQL"],
    "Среда": ["Анализ данных"],
}
TIPS = (
    "Индексы начинаются с 0.",
    "Кортеж нельзя изменить.",
    "while работает, пока условие истинно."
)
def tips_generator():
    for tip in TIPS:
        yield tip
tips = tips_generator()

# ---------- Команды ----------
@bot.message_handler(commands=["start"])
def handle_start(message):
    users.add(message.chat.id)
    bot.send_message(message.chat.id, "Привет! Я бот курсов.")
# Шаг 1. Строки и списки
@bot.message_handler(commands=["courses"])
def show_courses(message):
    text = "Курсы:\n" + "\n".join(courses)
    text += f"\nВсего: {len(courses)}, первый: {courses[0]}"
    bot.send_message(message.chat.id, text)
# Шаг 2. Кортеж
@bot.message_handler(commands=["contacts"])
def show_contacts(message):
    center, phone = CONTACTS
    bot.send_message(message.chat.id, f"{center}\nТелефон: {phone}")
# Шаг 3. Множество
@bot.message_handler(commands=["stats"])
def show_stats(message):
    bot.send_message(message.chat.id, f"Уникальных пользователей: {len(users)}")
# Шаг 4. Словарь и цикл for
@bot.message_handler(commands=["hours"])
def show_hours(message):
    text = "Длительность курсов:"
    for name, hours in HOURS.items():
        text += f"\n{name} — {hours} ч."
    bot.send_message(message.chat.id, text)
# Шаг 5. Вложенные структуры и вложенный цикл
@bot.message_handler(commands=["schedule"])
def show_schedule(message):
    text = "Расписание:"
    for day, lessons in SCHEDULE.items():
        text += f"\n{day}:"
        for lesson in lessons:
            text += f"\n {lesson}"
    bot.send_message(message.chat.id, text)
# Шаг 6. Цикл while
@bot.message_handler(commands=["countdown"])
def countdown(message):
    n = 5
    text = ""
    while n > 0:
        text += f"{n}\n"
        n -= 1
    bot.send_message(message.chat.id, text + "Старт!")
# Шаг 7. Генератор и next()
@bot.message_handler(commands=["tip"])
def show_tip(message):
    tip = next(tips, None)
    if tip is None:
        bot.send_message(message.chat.id, "Советы закончились.")
    else:
        bot.send_message(message.chat.id, tip)
# Шаг 8. Генератор списка
@bot.message_handler(commands=["search"])
def search(message):
    parts = message.text.split(maxsplit=1)
    if len(parts) < 2:
        bot.send_message(message.chat.id, "Использование: /search <слово>")
        return
    word = parts[1].strip().lower()
    found = [c for c in courses if word in c.lower()]
    if found:
        bot.send_message(message.chat.id, "Найдено: " + ", ".join(found))
    else:
        bot.send_message(message.chat.id, "Ничего не найдено.")
# Задание 1
@bot.message_handler(commands=["add"])
def add_course(message):
    parts = message.text.split(maxsplit=1)
    if len(parts) < 2:
        bot.send_message(
            message.chat.id,
            "Использование: /add <название>"
        )
        return
    course = parts[1].strip()
    courses.append(course)
    bot.send_message(
        message.chat.id,
        f"Курс добавлен: {course}"
    )
@bot.message_handler(func=lambda message: message.text.startswith("/last"))
def last_courses(message):
    try:
        count = int(message.text[5:])
        last_courses = courses[-count:]
        bot.send_message(
            message.chat.id,
            f"Последние {count} курса:\n" + "\n".join(last_courses))
    except (ValueError, IndexError):
        bot.send_message(
            message.chat.id,
            "Используй: /last2, /last5, /last10")
# Задание 2 (
@bot.message_handler(commands=["total"])
def total_hours(message):
    total = 0
    for hours in HOURS.values():
        total += hours
    bot.send_message(
        message.chat.id,
        f"Общее количество часов: {total}")
# Задание 3
@bot.message_handler(commands=["even"])
def even_while(message):
    parts = message.text.split(maxsplit=1)
    if len(parts) < 2:
        bot.send_message(
            message.chat.id,
            "Использование: /even <n>")
        return
    n = int(parts[1])
    i = 1
    even_numbers = []
    while i <= n:
        if i % 2 == 0:
            even_numbers.append(i)
        i += 1
    bot.send_message(
        message.chat.id,
        "Чётные числа (while): "
        + ", ".join(map(str, even_numbers))
    )
@bot.message_handler(commands=["evenlist"])
def even_list(message):
    parts = message.text.split(maxsplit=1)
    if len(parts) < 2:
        bot.send_message(
            message.chat.id,
            "Использование: /evenlist <n>"
        )
        return
    n = int(parts[1])
    even_numbers = [i for i in range(1, n + 1) if i % 2 == 0]
    bot.send_message(
        message.chat.id,
        "Чётные числа (генератор списка): "
        + ", ".join(map(str, even_numbers))
    )
# ---------- Обычный текст ----------
@bot.message_handler(content_types=["text"])
def handle_text(message):
    if message.text.startswith("/"):
        return
    text = message.text.lower().strip()
    if text == "привет":
        reply = "Здравствуйте! Введите /courses"
    else:
        reply = "Не понимаю. Команды: /courses /contacts /stats /hours /schedule /countdown /tip /search"
    bot.send_message(message.chat.id, reply)
bot.polling(none_stop=True)


In [11]:
list1 = []
for x in courses:
    if len(x) > 5:
        list1.append(x)

In [9]:
long_names = [c for c in courses if len(c) > 5]

In [10]:
long_names

['Python', 'Анализ данных', 'S3 Amazon']

In [12]:
list1

['Python', 'Анализ данных', 'S3 Amazon']